In [1]:
from collections.abc import Hashable
import os
from pathlib import Path

import pymimir

from graph_encoder import GraphEncoder

/home/dominik/miniconda3/envs/relationalgnn/lib/python3.14/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /home/dominik/miniconda3/envs/relationalgnn/lib/python3.14/site-packages/libpyg.so
  import torch_geometric.typing
/home/dominik/miniconda3/envs/relationalgnn/lib/python3.14/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /home/dominik/miniconda3/envs/relationalgnn/lib/python3.14/site-packages/libpyg.so
  import torch_geometric.typing
/home/dominik/miniconda3/envs/relationalgnn/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_path = Path(os.getcwd()).parent / "data"
d = pymimir.Domain(data_path / "domain.pddl")
p = pymimir.Problem(d, data_path / "test_clear_probBLOCKS-3-0.pddl", "grounded")

[LiftedGrounder] Number of fluent grounded atoms reachable in delete-free problem: 19
(on c a)
(on a c)
(on a a)
(on c c)
(on a b)
(handempty)
(on c b)
(clear c)
(clear b)
(ontable a)
(clear a)
(ontable c)
(on b a)
(ontable b)
(holding b)
(holding a)
(on b c)
(holding c)
(on b b)
Num axioms: 0
Num actions: 24


In [3]:
s = p.get_initial_state()
s

State(index=0, fluent_atoms=[(clear a), (clear b), (clear c), (handempty), (ontable a), (ontable b), (ontable c)], static_atoms=[(object a), (object b), (object c)], derived_atoms=[], fluent_numerics=[])

In [4]:
# help(pymimir.GroundAction)
a1 = pymimir.GroundAction.new(d.get_action("pickup"), [p.get_object("a")], p)
a1

(pickup a)

In [5]:
s1 = a1.apply(s)
s1

State(index=1, fluent_atoms=[(clear b), (clear c), (holding a), (ontable b), (ontable c)], static_atoms=[(object a), (object b), (object c)], derived_atoms=[], fluent_numerics=[])

In [6]:
s1

State(index=1, fluent_atoms=[(clear b), (clear c), (holding a), (ontable b), (ontable c)], static_atoms=[(object a), (object b), (object c)], derived_atoms=[], fluent_numerics=[])

In [7]:
a2 = pymimir.GroundAction.new(d.get_action("stack"), [p.get_object("a"), p.get_object("b")], p)
a2

(stack a b)

In [8]:
s2 = a2.apply(s1)
s2

State(index=2, fluent_atoms=[(clear a), (clear c), (handempty), (on a b), (ontable b), (ontable c)], static_atoms=[(object a), (object b), (object c)], derived_atoms=[], fluent_numerics=[])

In [9]:
s = s2

In [10]:
encoder = GraphEncoder(d, "obj")

In [11]:
encoder.predicate_arity_dict

{'on': 2,
 'ontable': 1,
 'clear': 1,
 'holding': 1,
 'on_G': 2,
 'ontable_G': 1,
 'clear_G': 1,
 'holding_G': 1,
 'not:on_G': 2,
 'not:ontable_G': 1,
 'not:clear_G': 1,
 'not:holding_G': 1}

In [12]:
g = encoder.encode(s, p.get_objects())
g

In [13]:
g.nodes.data()

NodeDataView({b: {'node_type': 'obj'}, a: {'node_type': 'obj'}, c: {'node_type': 'obj'}, (clear c): {'node_type': 'clear', 'class_type': <class 'pymimir.wrapper_formalism.GroundAtom'>}, (clear a): {'node_type': 'clear', 'class_type': <class 'pymimir.wrapper_formalism.GroundAtom'>}, (ontable c): {'node_type': 'ontable', 'class_type': <class 'pymimir.wrapper_formalism.GroundAtom'>}, (ontable b): {'node_type': 'ontable', 'class_type': <class 'pymimir.wrapper_formalism.GroundAtom'>}, (on a b): {'node_type': 'on', 'class_type': <class 'pymimir.wrapper_formalism.GroundAtom'>}, (clear a): {'node_type': 'clear_G', 'class_type': <class 'pymimir.wrapper_formalism.GroundLiteral'>}})

In [14]:
g.edges.data()

OutMultiEdgeDataView([(b, (ontable b), {'pos': 0}), (b, (on a b), {'pos': 1}), (a, (clear a), {'pos': 0}), (a, (on a b), {'pos': 0}), (a, (clear a), {'pos': 0}), (c, (clear c), {'pos': 0}), (c, (ontable c), {'pos': 0})])

In [15]:
for edge, data in g.edges.items():
    print(edge)
    print(data)
    print("========")

(b, (ontable b), 0)
{'pos': 0}
(b, (on a b), 0)
{'pos': 1}
(a, (clear a), 0)
{'pos': 0}
(a, (on a b), 0)
{'pos': 0}
(a, (clear a), 0)
{'pos': 0}
(c, (clear c), 0)
{'pos': 0}
(c, (ontable c), 0)
{'pos': 0}


In [16]:
src, dst, pos = edge

In [17]:
g.nodes[src]

{'node_type': 'obj'}

In [18]:
as_pyg = encoder.to_pyg(g)

In [19]:
as_pyg

HeteroData(
  obj_names=[3],
  obj={ x=[3] },
  ontable_G={ x=[0, 1] },
  clear_G={ x=[1, 1] },
  ontable={ x=[2, 1] },
  holding={ x=[0, 1] },
  not:clear_G={ x=[0, 1] },
  not:on_G={ x=[0, 2] },
  holding_G={ x=[0, 1] },
  not:holding_G={ x=[0, 1] },
  on_G={ x=[0, 2] },
  not:ontable_G={ x=[0, 1] },
  clear={ x=[2, 1] },
  on={ x=[1, 2] },
  (obj, 0, on)={ edge_index=[2, 2] },
  (obj, 1, on)={ edge_index=[2, 0] },
  (obj, 0, ontable)={ edge_index=[2, 2] },
  (obj, 0, clear)={ edge_index=[2, 2] },
  (obj, 0, holding)={ edge_index=[2, 0] },
  (obj, 0, on_G)={ edge_index=[2, 0] },
  (obj, 1, on_G)={ edge_index=[2, 0] },
  (obj, 0, ontable_G)={ edge_index=[2, 0] },
  (obj, 0, clear_G)={ edge_index=[2, 1] },
  (obj, 0, holding_G)={ edge_index=[2, 0] },
  (obj, 0, not:on_G)={ edge_index=[2, 0] },
  (obj, 1, not:on_G)={ edge_index=[2, 0] },
  (obj, 0, not:ontable_G)={ edge_index=[2, 0] },
  (obj, 0, not:clear_G)={ edge_index=[2, 0] },
  (obj, 0, not:holding_G)={ edge_index=[2, 0] },
  (on,

In [20]:
encoder.edge_types

[('obj', '0', 'on'),
 ('obj', '1', 'on'),
 ('obj', '0', 'ontable'),
 ('obj', '0', 'clear'),
 ('obj', '0', 'holding'),
 ('obj', '0', 'on_G'),
 ('obj', '1', 'on_G'),
 ('obj', '0', 'ontable_G'),
 ('obj', '0', 'clear_G'),
 ('obj', '0', 'holding_G'),
 ('obj', '0', 'not:on_G'),
 ('obj', '1', 'not:on_G'),
 ('obj', '0', 'not:ontable_G'),
 ('obj', '0', 'not:clear_G'),
 ('obj', '0', 'not:holding_G'),
 ('on', '0', 'obj'),
 ('on', '1', 'obj'),
 ('ontable', '0', 'obj'),
 ('clear', '0', 'obj'),
 ('holding', '0', 'obj'),
 ('on_G', '0', 'obj'),
 ('on_G', '1', 'obj'),
 ('ontable_G', '0', 'obj'),
 ('clear_G', '0', 'obj'),
 ('holding_G', '0', 'obj'),
 ('not:on_G', '0', 'obj'),
 ('not:on_G', '1', 'obj'),
 ('not:ontable_G', '0', 'obj'),
 ('not:clear_G', '0', 'obj'),
 ('not:holding_G', '0', 'obj')]

In [21]:
as_pyg[("obj", "0", "clear_G")]["edge_index"]

tensor([[0],
        [0]])

In [26]:
as_pyg["on_G"]

{'x': tensor([], size=(0, 2))}

In [23]:
from torch_geometric.nn.resolver import aggregation_resolver as aggr_resolver
aggr_resolver("sum")

SumAggregation()